# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. All references to dataset elements—such as record sets, fields, and columns—are made using their Croissant `@id` attributes, ensuring reproducibility and clarity.

### Dataset Source
This dataset is described using a Croissant schema, accessible at the URL given below.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load the FAIR² dataset metadata and available records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description from the metadata
print(f"Dataset Name: {dataset.metadata.name}\n\nDescription: {dataset.metadata.description}\n")

## 2. Data Overview
In Croissant, data is organized into one or more *record sets*. Each record set has a unique identifier (`@id`). Each record set defines its available fields and corresponding Croissant `@id`s. Let's list all available record sets, their `@id`s, and respective fields.

In [ ]:
# List record sets with their @id and field details
record_sets = dataset.metadata.record_sets

if not record_sets:
    print("No record sets found in dataset metadata.\nTrying to infer from distribution ...")
    # Some datasets may define the record set implicitly via distribution
    # Try dataset.records() without specifying record_set
    try:
        sample_record = next(dataset.records())
        print("Default/anonymous record set found. Sample record:")
        print(sample_record)
        record_set_ids = [None]
    except Exception as e:
        print(f"Could not iterate records: {e}")
        record_set_ids = []
else:
    for recset in record_sets:
        print(f"- Record Set Name: {getattr(recset, 'name', 'N/A')}")
        print(f"  Record Set @id: {recset.id}")
        # Fields information
        print("  Fields:")
        for field in recset.fields:
            print(f"    - Field Name: {getattr(field, 'name', 'N/A')}, Field @id: {field.id}, DataType: {getattr(field, 'data_type', 'N/A')}")
        print("")
    record_set_ids = [r.id for r in record_sets]

## 3. Data Extraction
We will extract data from the available record set(s). Each record set is referenced using its `@id` as shown above. Data is loaded into a pandas DataFrame, and columns (again referenced by their `@id`s) are shown for inspection.

In [ ]:
# Prepare to extract records from all found record sets
dataframes = {}

if not record_sets:
    print("Inferring records from default record set (None)...")
    records = list(dataset.records())
    df = pd.DataFrame(records)
    dataframes[None] = df
    print("Columns in extracted DataFrame:")
    print(df.columns.tolist())
    display(df.head())
else:
    for recset in record_sets:
        recset_id = recset.id
        records = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"Record set @id: {recset_id}")
        print("Columns (field @id) in extracted DataFrame:")
        print(df.columns.tolist())
        display(df.head())


## 4. Exploratory Data Analysis (EDA)
Here we demonstrate common EDA steps such as filtering records based on a numeric field, normalizing its values, and grouping records by another key field. All field references are made using their Croissant `@id`.

In [ ]:
# ---- Configuration: Select which record set and fields to analyze ----

# Helper: Find a numeric field in the first record set
if record_sets:
    selected_recset = record_sets[0]
    selected_recset_id = selected_recset.id
    
    # Try to auto-select a numeric field and a grouping field
    numeric_fields = [f for f in selected_recset.fields if getattr(f, 'data_type', None) in ('Integer', 'Float', 'Number')]
    group_fields = [f for f in selected_recset.fields if getattr(f, 'data_type', None) == 'Text']
    numeric_field_id = numeric_fields[0].id if numeric_fields else None
    group_field_id = group_fields[0].id if group_fields else None
else:
    selected_recset_id = None
    df = dataframes[None]
    # Heuristics: pick the first numeric-looking column
    sample_row = df.iloc[0]
    numeric_field_id = None
    group_field_id = None
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field_id = c
            break
    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]) and c != numeric_field_id:
            group_field_id = c
            break

# Select DataFrame
if selected_recset_id in dataframes:
    df = dataframes[selected_recset_id]
elif None in dataframes:
    df = dataframes[None]
else:
    print('No DataFrame loaded.')
    df = None

print(f"Chosen record set @id: {selected_recset_id if selected_recset_id else 'default'}")
print(f"Numeric field @id: {numeric_field_id}")
print(f"Group field @id: {group_field_id}\n")

# ----- Filtering, Normalizing, and Grouping -----
if df is not None and numeric_field_id and numeric_field_id in df.columns:
    # Filter: Keep only records where numeric value > threshold (10)
    threshold = 10
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
    print(f"Filtered records where [{numeric_field_id}] > {threshold}:")
    display(filtered_df.head())

    # Normalize numeric column (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
    print(f"\nNormalized [{numeric_field_id}] (z-score):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group and summarize by group_field
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id, dropna=True)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean [{numeric_field_id}] by [{group_field_id}]:")
        display(grouped_df.head())
else:
    print("Could not find numeric field for EDA.")

## 5. Visualization
Here we will visualize the distribution of the selected numeric field and its relation grouped by the selected group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id and numeric_field_id in df.columns:
    # Histogram of numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group field (if available)
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and process the FAIR² dataset on human protein mass spectrometry using the Croissant metadata schema and the `mlcroissant` library. All data fields and structures were referenced using their `@id`, ensuring reproducibility and alignment with FAIR principles. Such an approach enables seamless programmatic access and allows users to build data pipelines that are robust to schema changes and well-documented.

_For more information on Croissant and mlcroissant, see: https://mlcommons.org/croissant/ and https://github.com/mlcommons/croissant._